# 50 — Evaluate and compare vs old V6.2E YOLO11n@640
Baseline: mAP50 0.768 / mAP50:95 0.413 / gap 0.355.
Gate (legacy): global mAP50:95 regression ≤ 1 pt; foreign_object AP50 +2 pt and AP50:95 +1 pt;
broken_strand AP50:95 +1 pt. A pass only opens human review/promotion.

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
outcome = pipeline.run_stage("evaluate", force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
cfg = load_config("yolo11n_ablation")
cmp_csv = paths.exports / f"{cfg.get('training.experiment')}_vs_baseline.csv"
if cmp_csv.exists():
    display(pd.read_csv(cmp_csv))

If foreign_object improves but broken_strand regresses, split the ablation by source
(foreign-only vs strand-only) before freezing anything.